<a href="https://colab.research.google.com/github/riazhasan998/Data-Science/blob/master/GTSRB_(CNN_vs_InceptionV3_vs_ResNet50_vs_VGG16).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
meowmeowmeowmeowmeow_gtsrb_german_traffic_sign_path = kagglehub.dataset_download('meowmeowmeowmeowmeow/gtsrb-german-traffic-sign')

print('Data source import complete.')


Data source import complete.


# Traffic Sign Classification for Autonomous Vehicles

## Problem Introduction
Traffic sign recognition is critical for autonomous vehicles to navigate roads safely and comply with traffic regulations. Misinterpreting signs can lead to accidents, endangering lives. This project uses the GTSRB dataset to classify 43 types of German traffic signs, enabling vehicles to make real-time decisions. The dataset contains ~51,839 images, meeting the requirement of >20,000 images and >5 classes. Accurate classification can enhance road safety and support advanced driver-assistance systems (ADAS).

## Dataset Info:

The GTSRB dataset is loaded from CSV files containing image paths and labels. Images are resized to 32x32 pixels for consistency, and the dataset is split into training (70%), validation (15%), and testing (15%) sets.

## Import Libraries

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow import keras
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications import ResNet50V2, VGG16, VGG19, MobileNetV2
from tensorflow.keras.applications.resnet_v2 import preprocess_input as preprocess_resnet50v2
from tensorflow.keras.applications.vgg16 import preprocess_input as preprocess_vgg16
from tensorflow.keras.applications.vgg19 import preprocess_input as preprocess_vgg19
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mobilenetv2
from tensorflow.keras.utils import to_categorical

In [ ]:
trian_path = r'/kaggle/input/gtsrb-german-traffic-sign/Train'

# Get all image paths
images = []
for folder in os.listdir(trian_path):
    path = os.path.join(trian_path,folder)
    for img in os.listdir(path):
        if img.endswith(('.jpg', '.png', '.webp')):
            images.append(os.path.join(path, img))


In [ ]:
random_images = random.sample(images, 16)

In [ ]:
plt.figure(figsize=(12, 12))
for i, img_path in enumerate(tqdm(random_images)):
    img = cv2.imread(img_path)
    if img is None:
        continue
    img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    plt.subplot(4, 4, i + 1)
    plt.imshow(img,cmap='gray')
    plt.axis('off')
    plt.title(f'Image {i+1}')

plt.tight_layout()
plt.show()

In [ ]:
import os, shutil
import numpy as np
from sklearn.model_selection import train_test_split

original_dataset_dir = '/kaggle/input/gtsrb-german-traffic-sign/Train'
base_dir = '/kaggle/working/split_data'

train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Split each class
for class_name in os.listdir(original_dataset_dir):
    class_path = os.path.join(original_dataset_dir, class_name)
    if not os.path.isdir(class_path): continue

    images = os.listdir(class_path)
    train_files, temp_files = train_test_split(images, test_size=0.4, random_state=42)
    val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)

    for subset_name, subset_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
        subset_class_dir = os.path.join(base_dir, subset_name, class_name)
        os.makedirs(subset_class_dir, exist_ok=True)
        for fname in subset_files:
            src = os.path.join(class_path, fname)
            dst = os.path.join(subset_class_dir, fname)
            shutil.copy(src, dst)


In [ ]:
def cnn_preprocess(x):
    return x/255.0

In [ ]:
IMG_SIZE = 224
batch_size = 32

## Data Preparation Phase

# CNN Model

## CNN Data Generators

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

cnn_datagen = ImageDataGenerator(preprocessing_function=cnn_preprocess)

cnn_train_generator = cnn_datagen.flow_from_directory(
    '/kaggle/working/split_data/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=batch_size,
    class_mode='categorical'
)

cnn_val_generator = cnn_datagen.flow_from_directory(
    '/kaggle/working/split_data/val',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=batch_size,
    class_mode='categorical'
)

cnn_test_generator = cnn_datagen.flow_from_directory(
    '/kaggle/working/split_data/test',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)


## CNN Modeling

In [ ]:
cnn_model = keras.Sequential([
    keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    keras.layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),

    keras.layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),

    keras.layers.Conv2D(filters=128, kernel_size=(3, 3), activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),

    keras.layers.Flatten(),
    keras.layers.Dropout(0.4),

    keras.layers.Dense(units=128, activation='relu'),
    keras.layers.Dense(units=64, activation='relu'),
    keras.layers.Dense(units=64, activation='relu'),

    keras.layers.Dense(units=len(cnn_val_generator.class_indices), activation='softmax', dtype='float32')
])

In [ ]:
cnn_model.summary()

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),

    ModelCheckpoint(
        filepath='CNN_best_model.keras',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=False,
        mode='min',
        verbose=1
    )
]

In [ ]:
cnn_model.compile(optimizer=keras.optimizers.Adam(),
              loss=keras.losses.CategoricalCrossentropy(),
               metrics=['accuracy',keras.metrics.Precision(name='precision'),keras.metrics.Recall(name='recall')
])

In [ ]:
cnn_history = cnn_model.fit(
        cnn_train_generator,
        epochs = 20,
        validation_data = cnn_val_generator,
        callbacks = callbacks
        )

In [ ]:
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(cnn_history.history['accuracy'],label='train accuracy',marker='o',color='blue')
plt.plot(cnn_history.history['val_accuracy'],label='val accuracy',marker='o',color='orange')
plt.title('Accuracy Over Epochs ( cnn_model )')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(cnn_history.history['loss'], label='Train Loss', marker='o')
plt.plot(cnn_history.history['val_loss'], label='Val Loss', marker='o')
plt.title('Loss Over Epochs ( cnn_model )')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()


## vgg16_model

## VGG16 Data Generators

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

vgg_datagen = ImageDataGenerator(preprocessing_function=cnn_preprocess)

vgg_train_generator = vgg_datagen.flow_from_directory(
    '/kaggle/working/split_data/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=batch_size,
    class_mode='categorical'
)

vgg_val_generator = vgg_datagen.flow_from_directory(
    '/kaggle/working/split_data/val',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=batch_size,
    class_mode='categorical'
)

vgg_test_generator = vgg_datagen.flow_from_directory(
    '/kaggle/working/split_data/test',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)


## VGG16 Modeling

In [ ]:
vgg_traind_layer=VGG16(weights='imagenet',include_top=False,input_shape=(IMG_SIZE,IMG_SIZE,3))

In [ ]:
for layer in vgg_traind_layer.layers:
    layer.trainable=False


In [ ]:
VGG_model=keras.Sequential([
   vgg_traind_layer,
   keras.layers.Flatten(),
   keras.layers.Dense(units=256,activation='relu'),
   keras.layers.Dense(units=len(vgg_val_generator.class_indices),activation='softmax')

])

In [ ]:
VGG_model.summary()

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),

    ModelCheckpoint(
        filepath='VGG_best_model.keras',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=False,
        mode='min',
        verbose=1
    )
]

In [ ]:
VGG_model.compile(optimizer=keras.optimizers.Adam(),
              loss=keras.losses.CategoricalCrossentropy(),
               metrics=['accuracy',keras.metrics.Precision(name='precision'),keras.metrics.Recall(name='recall')
])

In [ ]:
VGG_history = VGG_model.fit(
        vgg_train_generator,
        epochs = 20,
        validation_data = vgg_val_generator,
        callbacks = callbacks
        )

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(VGG_history.history['loss'], label='Train Loss', marker='o', color='blue')
plt.plot(VGG_history.history['val_loss'], label='Val Loss', marker='o', color='orange')
plt.title('Loss Over Epochs (vgg16)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# 📈 Accuracy
plt.subplot(1, 2, 2)
plt.plot(VGG_history.history['accuracy'], label='Train Accuracy', marker='o', color='red')
plt.plot(VGG_history.history['val_accuracy'], label='Val Accuracy', marker='o', color='green')
plt.title('Accuracy Over Epochs (vgg16) ')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

## InceptionV3_model

# InceptionV3 Data Generators

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

Inception_datagen = ImageDataGenerator(preprocessing_function=cnn_preprocess)

Inception_train_generator = Inception_datagen.flow_from_directory(
    '/kaggle/working/split_data/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=batch_size,
    class_mode='categorical'
)

Inception_val_generator = Inception_datagen.flow_from_directory(
    '/kaggle/working/split_data/val',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=batch_size,
    class_mode='categorical'
)

Inception_test_generator = Inception_datagen.flow_from_directory(
    '/kaggle/working/split_data/test',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)


# InceptionV3 Modeling

In [ ]:
from tensorflow.keras.applications import InceptionV3
trained_Inception_layers = InceptionV3(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

In [ ]:
for layer in trained_Inception_layers.layers:
    layer.trainable = False

In [ ]:
Inception_model = tf.keras.models.Sequential([
    trained_Inception_layers,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(len(Inception_val_generator.class_indices), activation='softmax')
])

In [ ]:
Inception_model.summary()

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        filepath='Inception_best_model.keras',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=False,
        mode='min',
        verbose=1
    )
]

In [ ]:
Inception_model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

In [ ]:
Inception_history = Inception_model.fit(
    Inception_train_generator,
    epochs=20,
    validation_data=Inception_val_generator,
    callbacks=callbacks
)

In [ ]:
plt.figure(figsize=(12, 5))

# Loss
plt.subplot(1, 2, 1)
plt.plot(Inception_history.history['loss'], label='Train Loss', marker='o', color='blue')
plt.plot(Inception_history.history['val_loss'], label='Val Loss', marker='o', color='orange')
plt.title('Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(Inception_history.history['accuracy'], label='Train Accuracy', marker='o', color='red')
plt.plot(Inception_history.history['val_accuracy'], label='Val Accuracy', marker='o', color='green')
plt.title('Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

## ResNet_model

## ResNet50V2 Data Generators

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

ResNet_datagen = ImageDataGenerator(preprocessing_function=cnn_preprocess)

ResNet_train_generator = ResNet_datagen.flow_from_directory(
    '/kaggle/working/split_data/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=batch_size,
    class_mode='categorical'
)

ResNet_val_generator = ResNet_datagen.flow_from_directory(
    '/kaggle/working/split_data/val',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=batch_size,
    class_mode='categorical'
)

ResNet_test_generator = ResNet_datagen.flow_from_directory(
    '/kaggle/working/split_data/test',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)


## ResNet50V2 Modeling

In [ ]:
trained_ResNet_layers = ResNet50V2(weights='imagenet',include_top=False,input_shape=(IMG_SIZE,IMG_SIZE,3))

In [ ]:
for layer in trained_ResNet_layers.layers:
    layer.trainable = False


In [ ]:
ResNet_model = keras.models.Sequential([
    trained_ResNet_layers,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dense(len(ResNet_val_generator.class_indices), activation='softmax')
])

In [ ]:
ResNet_model.summary()

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),

    ModelCheckpoint(
        filepath='ResNet_best_model.keras',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=False,
        mode='min',
        verbose=1
    )
]

In [ ]:
ResNet_model.compile(optimizer=keras.optimizers.Adam(),
              loss=keras.losses.CategoricalCrossentropy(),
               metrics=['accuracy',keras.metrics.Precision(name='precision'),keras.metrics.Recall(name='recall')
])

In [ ]:
ResNet_history = ResNet_model.fit(
        ResNet_train_generator,
        epochs = 20,
        validation_data = ResNet_val_generator,
        callbacks = callbacks
        )

In [ ]:
plt.figure(figsize=(12, 5))

# 📉 Loss
plt.subplot(1, 2, 1)
plt.plot(ResNet_history.history['loss'], label='Train Loss', marker='o', color='blue')
plt.plot(ResNet_history.history['val_loss'], label='Val Loss', marker='o', color='orange')
plt.title('Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# 📈 Accuracy
plt.subplot(1, 2, 2)
plt.plot(ResNet_history.history['accuracy'], label='Train Accuracy', marker='o', color='red')
plt.plot(ResNet_history.history['val_accuracy'], label='Val Accuracy', marker='o', color='green')
plt.title('Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

## mobile_model

# MobileNetV2 Data Generators

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

Mobile_datagen = ImageDataGenerator(preprocessing_function=cnn_preprocess)

Mobile_train_generator = Mobile_datagen.flow_from_directory(
    '/kaggle/working/split_data/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=batch_size,
    class_mode='categorical'
)

Mobile_val_generator = Mobile_datagen.flow_from_directory(
    '/kaggle/working/split_data/val',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=batch_size,
    class_mode='categorical'
)

Mobile_test_generator = Mobile_datagen.flow_from_directory(
    '/kaggle/working/split_data/test',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)


# MobileNetV2 Modeling

In [ ]:
trained_mobile_layers = ResNet50V2(weights='imagenet',include_top=False,input_shape=(IMG_SIZE,IMG_SIZE,3))

In [ ]:
for layer in trained_mobile_layers.layers:
    layer.trainable = False


In [ ]:
mobile_model = keras.models.Sequential([
    trained_ResNet_layers,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dense(len(Mobile_val_generator.class_indices), activation='softmax')
])

In [ ]:
mobile_model.summary()

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),

    ModelCheckpoint(
        filepath='mobile_best_model.keras',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=False,
        mode='min',
        verbose=1
    )
]

In [ ]:
mobile_model.compile(optimizer=keras.optimizers.Adam(),
              loss=keras.losses.CategoricalCrossentropy(),
               metrics=['accuracy',keras.metrics.Precision(name='precision'),keras.metrics.Recall(name='recall')
])

In [ ]:
mobile_history = ResNet_model.fit(
        Mobile_train_generator,
        epochs = 20,
        validation_data = Mobile_val_generator,
        callbacks = callbacks
        )

In [ ]:
plt.figure(figsize=(12, 5))

# 📉 Loss
plt.subplot(1, 2, 1)
plt.plot(mobile_history.history['loss'], label='Train Loss', marker='o', color='blue')
plt.plot(mobile_history.history['val_loss'], label='Val Loss', marker='o', color='orange')
plt.title('Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# 📈 Accuracy
plt.subplot(1, 2, 2)
plt.plot(mobile_history.history['accuracy'], label='Train Accuracy', marker='o', color='red')
plt.plot(mobile_history.history['val_accuracy'], label='Val Accuracy', marker='o', color='green')
plt.title('Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()


## Compare Between Models

In [ ]:
history = {
    "CNN": {
        "accuracy": cnn_history.history['accuracy'],
        "val_accuracy": cnn_history.history['val_accuracy'],
        "loss": cnn_history.history['loss'],
        "val_loss": cnn_history.history['val_loss']
    },
    "VGG16": {
        "accuracy": VGG_history.history['accuracy'],
        "val_accuracy": VGG_history.history['val_accuracy'],
        "loss": VGG_history.history['loss'],
        "val_loss": VGG_history.history['val_loss']
    },
    "Inception_history": {
        "accuracy": Inception_history.history['accuracy'],
        "val_accuracy": Inception_history.history['val_accuracy'],
        "loss": Inception_history.history['loss'],
        "val_loss": Inception_history.history['val_loss']
    },
    "ResNet50V2": {
        "accuracy": ResNet_history.history['accuracy'],
        "val_accuracy": ResNet_history.history['val_accuracy'],
        "loss": ResNet_history.history['loss'],
        "val_loss": ResNet_history.history['val_loss']
    },
    "MobileNetV2": {
        "accuracy": mobile_history.history['accuracy'],
        "val_accuracy": mobile_history.history['val_accuracy'],
        "loss": mobile_history.history['loss'],
        "val_loss": mobile_history.history['val_loss']
    }
}


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
for model, values in history.items():
    plt.plot(values['val_accuracy'], label=model)
plt.title('Model Validation Accuracy Comparison')
plt.ylabel('Validation Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
for model, values in history.items():
    plt.plot(values['val_loss'], label=model)
plt.title('Model Validation Loss Comparison')
plt.ylabel('Validation Loss')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(15, 20))

sorted_models = sorted(history.items(), key=lambda x: min(x[1]['val_loss']))

for i, (model_name, model_history) in enumerate(sorted_models, 1):
    plt.subplot(4, 2, i)
    plt.plot(model_history['loss'], label='Train Loss')
    plt.plot(model_history['val_loss'], label='Validation Loss')


    min_val_loss_epoch = model_history['val_loss'].index(min(model_history['val_loss']))
    min_val_loss = min(model_history['val_loss'])

    plt.scatter(min_val_loss_epoch, min_val_loss, color='red', label=f'Best Epoch: {min_val_loss_epoch}\nBest Val Loss: {min_val_loss:.4f}')

    plt.title(model_name)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
best_CNN = load_model('/kaggle/working/CNN_best_model.keras')
best_MobileNetV2 = load_model('/kaggle/working/mobile_best_model.keras')
best_VGG16 = load_model('/kaggle/working/VGG_best_model.keras')
best_Inception = load_model('/kaggle/working/Inception_best_model.keras')
best_ResNet50 = load_model('/kaggle/working/ResNet_best_model.keras')

In [ ]:
models = ['Best CNN','Best MobileNetV2','Best VGG16','Best Inception','Best ResNest50']
best_train_accuracy= [
                        cnn_history.history['accuracy'][np.argmax(cnn_history.history['accuracy'])],
                        mobile_history.history['accuracy'][np.argmax(mobile_history.history['accuracy'])],
                        VGG_history.history['accuracy'][np.argmax(VGG_history.history['accuracy'])],
                        Inception_history.history['accuracy'][np.argmax(Inception_history.history['accuracy'])],
                        ResNet_history.history['accuracy'][np.argmax(ResNet_history.history['accuracy'])]
                     ]
best_val_accuracy= [
                        cnn_history.history['val_accuracy'][np.argmax(cnn_history.history['val_accuracy'])],
                        mobile_history.history['val_accuracy'][np.argmax(mobile_history.history['val_accuracy'])],
                        VGG_history.history['val_accuracy'][np.argmax(VGG_history.history['val_accuracy'])],
                        Inception_history.history['val_accuracy'][np.argmax(Inception_history.history['val_accuracy'])],
                        ResNet_history.history['val_accuracy'][np.argmax(ResNet_history.history['val_accuracy'])]
                     ]

In [ ]:
barWidth = 0.20

r1 = np.arange(len(best_train_accuracy))
r2 = [x + barWidth for x in r1]

plt.figure(figsize=(20, 12))
bars1 = plt.bar(r1, best_train_accuracy, color='#66B2FF', width=barWidth, edgecolor='grey', label='Train accuracy')
bars2 = plt.bar(r2, best_val_accuracy, color='#FF6666', width=barWidth, edgecolor='grey', label='Val accuracy')

plt.xlabel('Models', fontweight='bold')
plt.ylabel('Accuracy', fontweight='bold')
plt.xticks([r + barWidth/2 for r in range(len(best_train_accuracy))], models)

def add_labels(bars):
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2, height, f'{height:.2f}', ha='center', va='bottom')

add_labels(bars1)
add_labels(bars2)

plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.title('Model Accuracy vs Validation Accuracy Comparison')
plt.show()

## model had the best validation accuracy

In [ ]:
import pandas as pd
best_val_accuracy= [
                        cnn_history.history['val_accuracy'][np.argmax(cnn_history.history['val_accuracy'])],
                        mobile_history.history['val_accuracy'][np.argmax(mobile_history.history['val_accuracy'])],
                        VGG_history.history['accuracy'][np.argmax(VGG_history.history['val_accuracy'])],
                        Inception_history.history['val_accuracy'][np.argmax(Inception_history.history['val_accuracy'])],
                        ResNet_history.history['val_accuracy'][np.argmax(ResNet_history.history['val_accuracy'])]
                     ]
best_val_accuracy = pd.DataFrame({
    'Model': ['CNN', 'MobileNet', 'VGG', 'Inception', 'ResNet'],
    'val_accuracy': best_val_accuracy
})

# Sort by validation accuracy in descending order and get the best model
best_model_name = best_val_accuracy.sort_values(by='val_accuracy', ascending=False).iloc[0]['Model']
print(f"\nBest Model: {best_model_name}")

## model showed more stable training

In [ ]:
import numpy as np


histories = {
    'CNN': cnn_history,
    'MobileNet': mobile_history,
    'VGG': VGG_history,
    'Inception': Inception_history,
    'ResNet': ResNet_history
}


stability_metrics = []


for model_name, history in histories.items():

    train_loss = history.history['loss']


    loss_diffs = np.diff(train_loss)
    std_loss_diff = np.std(np.abs(loss_diffs))

    stability_metrics.append({
        'Model': model_name,
        'Std_Loss_Diff': std_loss_diff
    })

stability_df = pd.DataFrame(stability_metrics)

most_stable_model = stability_df.sort_values(by='Std_Loss_Diff', ascending=True).iloc[0]['Model']
print(f"Most Stable Model: {most_stable_model}")

##  overfitting observed

In [ ]:

histories = {
    'CNN': cnn_history,
    'MobileNet': mobile_history,
    'VGG': VGG_history,
    'Inception': Inception_history,
    'ResNet': ResNet_history
}


ACCURACY_GAP_THRESHOLD = 0.1  #  10% gap --> Overfitting
LOSS_GAP_THRESHOLD = 0.5      # 50 % gap --> Overfitting


overfitting_results = []


for model_name, history in histories.items():

    final_train_acc = history.history['accuracy'][-1]
    final_val_acc = history.history['val_accuracy'][-1]


    final_train_loss = history.history['loss'][-1]
    final_val_loss = history.history['val_loss'][-1]


    acc_gap = final_train_acc - final_val_acc
    loss_gap = final_val_loss - final_train_loss


    is_overfitting = False
    if acc_gap > ACCURACY_GAP_THRESHOLD or loss_gap > LOSS_GAP_THRESHOLD:
        is_overfitting = True


    overfitting_results.append({
        'Model': model_name,
        'Accuracy_Gap': acc_gap,
        'Loss_Gap': loss_gap,
        'Overfitting': 'Yes' if is_overfitting else 'No'
    })


overfitting_df = pd.DataFrame(overfitting_results)


print("\nOverfitting Analysis:")
print(overfitting_df[['Model', 'Accuracy_Gap','Loss_Gap', 'Overfitting']])


overfitting_models = overfitting_df[overfitting_df['Overfitting'] == 'Yes']['Model'].tolist()
if overfitting_models:
    print(f"\nModels showing overfitting: {', '.join(overfitting_models)}")
else:
    print("\nNo models show significant overfitting.")

## read test data

In [ ]:
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os


data_dir = '/kaggle/input/gtsrb-german-traffic-sign'
test_csv = os.path.join(data_dir, 'Test.csv')
IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 32
NUM_CLASSES = 43


test_df = pd.read_csv(test_csv)
test_df['Path'] = test_df['Path'].apply(lambda x: os.path.join(data_dir, x))
test_df['ClassId'] = test_df['ClassId'].astype(str)


test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='Path',
    y_col='ClassId',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


## CNN Evaluation

In [ ]:
best_CNN = load_model('/kaggle/working/CNN_best_model.keras')
with tf.device('/GPU:0'):
    test_loss , test_accuracy , test_precision , test_recall = best_CNN.evaluate(cnn_test_generator , verbose=1)
    print(f'Test Loss : {test_loss} - Test Accuracy : {test_accuracy} - Test Precision : {test_precision} - Test Recall : {test_recall}')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


y_true = cnn_test_generator.classes


y_pred_probs = best_CNN.predict(cnn_test_generator)
y_pred = np.argmax(y_pred_probs, axis=1)


cm = confusion_matrix(y_true, y_pred)


class_names = list(cnn_test_generator.class_indices.keys())


plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix FOR CNN')
plt.show()

## MobileNeT Evaluation

In [ ]:
best_MobileNetV2 = load_model('/kaggle/working/mobile_best_model.keras')
with tf.device('/GPU:0'):
    test_loss , test_accuracy , test_precision , test_recall = best_MobileNetV2.evaluate(Mobile_test_generator , verbose=1)
    print(f'Test Loss : {test_loss} - Test Accuracy : {test_accuracy} - Test Precision : {test_precision} - Test Recall : {test_recall}')

In [ ]:


y_true = Mobile_test_generator.classes


y_pred_probs = best_MobileNetV2.predict(Mobile_test_generator)
y_pred = np.argmax(y_pred_probs, axis=1)


cm = confusion_matrix(y_true, y_pred)


class_names = list(Mobile_test_generator.class_indices.keys())


plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix FOR MobileNe')
plt.show()

##  VGG16 Evaluation

In [ ]:
best_VGG16 = load_model('/kaggle/working/VGG_best_model.keras')
with tf.device('/GPU:0'):
    test_loss , test_accuracy , test_precision , test_recall = best_VGG16.evaluate(vgg_test_generator , verbose=1)
    print(f'Test Loss : {test_loss} - Test Accuracy : {test_accuracy} - Test Precision : {test_precision} - Test Recall : {test_recall}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


y_true = vgg_test_generator.classes


y_pred_probs = best_VGG16.predict(vgg_test_generator)
y_pred = np.argmax(y_pred_probs, axis=1)


cm = confusion_matrix(y_true, y_pred)


class_names = list(vgg_test_generator.class_indices.keys())


plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix FOR best_VGG16')
plt.show()

## Inception Evaluation

In [ ]:
best_Inception = load_model('/kaggle/working/Inception_best_model.keras')
with tf.device('/GPU:0'):
    test_loss , test_accuracy , test_precision , test_recall = best_Inception.evaluate(Inception_test_generator , verbose=1)
    print(f'Test Loss : {test_loss} - Test Accuracy : {test_accuracy} - Test Precision : {test_precision} - Test Recall : {test_recall}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


y_true = Inception_test_generator.classes


y_pred_probs = best_Inception.predict(Inception_test_generator)
y_pred = np.argmax(y_pred_probs, axis=1)


cm = confusion_matrix(y_true, y_pred)


class_names = list(Inception_test_generator.class_indices.keys())


plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix FOR MobileNe')
plt.show()

## ResNet50 Evaluation

In [ ]:
best_ResNet50 = load_model('/kaggle/working/ResNet_best_model.keras')
with tf.device('/GPU:0'):
    test_loss , test_accuracy , test_precision , test_recall = best_ResNet50.evaluate(ResNet_test_generator , verbose=1)
    print(f'Test Loss : {test_loss} - Test Accuracy : {test_accuracy} - Test Precision : {test_precision} - Test Recall : {test_recall}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


y_true = ResNet_test_generator.classes


y_pred_probs = best_ResNet50.predict(ResNet_test_generator)
y_pred = np.argmax(y_pred_probs, axis=1)


cm = confusion_matrix(y_true, y_pred)


class_names = list(ResNet_test_generator.class_indices.keys())

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix FOR MobileNe')
plt.show()

# Prediction Visualization

## CNN

In [ ]:
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os


data_dir = '/kaggle/input/gtsrb-german-traffic-sign'
test_csv = os.path.join(data_dir, 'Test.csv')
IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 32
NUM_CLASSES = 43


test_df = pd.read_csv(test_csv)
test_df['Path'] = test_df['Path'].apply(lambda x: os.path.join(data_dir, x))
test_df['ClassId'] = test_df['ClassId'].astype(str)


test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='Path',
    y_col='ClassId',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


In [ ]:
# Class dictionary
classes = {
    0: 'Speed limit (20km/h)', 1: 'Speed limit (30km/h)', 2: 'Speed limit (50km/h)',
    3: 'Speed limit (60km/h)', 4: 'Speed limit (70km/h)', 5: 'Speed limit (80km/h)',
    6: 'End of speed limit (80km/h)', 7: 'Speed limit (100km/h)', 8: 'Speed limit (120km/h)',
    9: 'No passing', 10: 'No passing veh over 3.5 tons', 11: 'Right-of-way at intersection',
    12: 'Priority road', 13: 'Yield', 14: 'Stop', 15: 'No vehicles',
    16: 'Veh > 3.5 tons prohibited', 17: 'No entry', 18: 'General caution',
    19: 'Dangerous curve left', 20: 'Dangerous curve right', 21: 'Double curve',
    22: 'Bumpy road', 23: 'Slippery road', 24: 'Road narrows on the right',
    25: 'Road work', 26: 'Traffic signals', 27: 'Pedestrians', 28: 'Children crossing',
    29: 'Bicycles crossing', 30: 'Beware of ice/snow', 31: 'Wild animals crossing',
    32: 'End speed + passing limits', 33: 'Turn right ahead', 34: 'Turn left ahead',
    35: 'Ahead only', 36: 'Go straight or right', 37: 'Go straight or left',
    38: 'Keep right', 39: 'Keep left', 40: 'Roundabout mandatory',
    41: 'End of no passing', 42: 'End no passing veh > 3.5 tons'
}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

num_images = 9
predictions = best_CNN.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

images, _ = next(test_generator)
test_generator.reset()

plt.figure(figsize=(15, 10))
for i in range(num_images):
    plt.subplot(3, 3, i+1)
    plt.imshow(images[i])
    plt.axis('off')

    true_label = class_labels[true_classes[i]]
    predicted_label = class_labels[predicted_classes[i]]
    confidence = np.max(predictions[i]) * 100

    title_color = 'green' if true_label == predicted_label else 'red'
    plt.title(f'True: {true_label}\nPred: {predicted_label}', color=title_color)

plt.suptitle("Prediction Visualization", fontsize=16)
plt.tight_layout()
plt.show()


## VGG16

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

num_images = 9
predictions = best_CNN.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

images, _ = next(test_generator)
test_generator.reset()

plt.figure(figsize=(15, 10))
for i in range(num_images):
    plt.subplot(3, 3, i+1)
    plt.imshow(images[i])
    plt.axis('off')

    true_label = class_labels[true_classes[i]]
    predicted_label = class_labels[predicted_classes[i]]
    confidence = np.max(predictions[i]) * 100

    title_color = 'green' if true_label == predicted_label else 'red'
    plt.title(f'True: {true_label}\nPred: {predicted_label}', color=title_color)

plt.suptitle("Prediction Visualization", fontsize=16)
plt.tight_layout()
plt.show()


## Inception

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

num_images = 9
predictions = best_Inception.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

images, _ = next(test_generator)
test_generator.reset()

plt.figure(figsize=(15, 10))
for i in range(num_images):
    plt.subplot(3, 3, i+1)
    plt.imshow(images[i])
    plt.axis('off')

    true_label = class_labels[true_classes[i]]
    predicted_label = class_labels[predicted_classes[i]]
    confidence = np.max(predictions[i]) * 100

    title_color = 'green' if true_label == predicted_label else 'red'
    plt.title(f'True: {true_label}\nPred: {predicted_label}', color=title_color)

plt.suptitle("Prediction Visualization", fontsize=16)
plt.tight_layout()
plt.show()


## MobileNeT

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

num_images = 9
predictions = best_CNN.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

images, _ = next(test_generator)
test_generator.reset()

plt.figure(figsize=(15, 10))
for i in range(num_images):
    plt.subplot(3, 3, i+1)
    plt.imshow(images[i])
    plt.axis('off')

    true_label = class_labels[true_classes[i]]
    predicted_label = class_labels[predicted_classes[i]]
    confidence = np.max(predictions[i]) * 100

    title_color = 'green' if true_label == predicted_label else 'red'
    plt.title(f'True: {true_label}\nPred: {predicted_label}', color=title_color)

plt.suptitle("Prediction Visualization", fontsize=16)
plt.tight_layout()
plt.show()


## ResNet50

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

num_images = 9
predictions = best_ResNet50.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

images, _ = next(test_generator)
test_generator.reset()

plt.figure(figsize=(15, 10))
for i in range(num_images):
    plt.subplot(3, 3, i+1)
    plt.imshow(images[i])
    plt.axis('off')

    true_label = class_labels[true_classes[i]]
    predicted_label = class_labels[predicted_classes[i]]
    confidence = np.max(predictions[i]) * 100

    title_color = 'green' if true_label == predicted_label else 'red'
    plt.title(f'True: {true_label}\nPred: {predicted_label}', color=title_color)

plt.suptitle("Prediction Visualization", fontsize=16)
plt.tight_layout()
plt.show()
